In [8]:
tabla='cmtse10'
dim='dim_tipseg'

In [9]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [10]:

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM cmtse10", con=connection2)

connection2.close()




In [11]:
base2

,tipsegcod,tipsegdes,indpagprescod,tipsegcalvbp,indotorcittcod,estregcod,indplansaluflg,tipsegdecurgflg,tipsegusucrecod,tipsegcrefec,tipsegusumodcod,tipsegmodfec
0,01,OBLIGATORIO,0,1,1,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
1,02,FACULTATIVO INDEPENDIENTE,0,1,0,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
2,03,CONTINUADOR FACULTATIVO,0,1,0,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
3,04,TRABAJADOR DEL HOGAR,0,1,1,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
4,05,AMA DE CASA,0,1,0,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
5,06,AGRARIO INDEPENDIENTE,0,1,1,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
6,07,PENSIONISTA,0,1,0,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
7,12,TERCERO,1,0,0,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
8,15,CHOFER PROFESIONAL,0,1,0,1,0,0,ADMIN,2017-05-13 19:30:03,None,None
9,19,CONSTRUCCION CIVIL,0,1,1,1,0,0,ADMIN,2017-05-13 19:30:03,None,None


In [12]:
base2.columns

Index(['tipsegcod', 'tipsegdes', 'indpagprescod', 'tipsegcalvbp',
       'indotorcittcod', 'estregcod', 'indplansaluflg', 'tipsegdecurgflg',
       'tipsegusucrecod', 'tipsegcrefec', 'tipsegusumodcod', 'tipsegmodfec'],
      dtype='object')

In [13]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM cmtse10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cmtse10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cmtse10 ()')
base2.to_sql(name='tmp_cmtse10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmtse10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cmtse10 
ALTER COLUMN tipsegcod TYPE character(2),
ALTER COLUMN tipsegdes TYPE character(40),
ALTER COLUMN indpagprescod TYPE character(1),
ALTER COLUMN tipsegcalvbp TYPE character(1),
ALTER COLUMN indotorcittcod TYPE character(1),
ALTER COLUMN estregcod TYPE character(1),
ALTER COLUMN indplansaluflg TYPE character(1),
ALTER COLUMN tipsegdecurgflg TYPE character(1),
ALTER COLUMN tipsegusucrecod TYPE character(10),
ALTER COLUMN tipsegcrefec TYPE date USING tipsegcrefec::date,
ALTER COLUMN tipsegmodfec TYPE date USING tipsegmodfec::date,
ALTER COLUMN tipsegusumodcod TYPE character(10);



UPDATE cmtse10 
SET 
tipsegdes=t.tipsegdes,
indpagprescod=t.indpagprescod,
tipsegcalvbp=t.tipsegcalvbp,
indotorcittcod=t.indotorcittcod,
estregcod=t.estregcod,
indplansaluflg=t.indplansaluflg,
tipsegdecurgflg=t.tipsegdecurgflg,
tipsegusucrecod=t.tipsegusucrecod,
tipsegcrefec=t.tipsegcrefec,
tipsegusumodcod=t.tipsegusumodcod,
tipsegmodfec=t.tipsegmodfec

FROM tmp_cmtse10 t 
WHERE cmtse10.tipsegcod = t.tipsegcod and cmtse10.tipsegcod IS NOT NULL;


INSERT INTO cmtse10 
(tipsegcod,tipsegdes,indpagprescod,tipsegcalvbp,indotorcittcod,estregcod,indplansaluflg,tipsegdecurgflg,tipsegusucrecod,tipsegcrefec,tipsegusumodcod,tipsegmodfec) 
SELECT 
tipsegcod,tipsegdes,indpagprescod,tipsegcalvbp,indotorcittcod,estregcod,indplansaluflg,tipsegdecurgflg,tipsegusucrecod,tipsegcrefec,tipsegusumodcod,tipsegmodfec

FROM tmp_cmtse10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmtse10 
    WHERE cmtse10.tipsegcod = t.tipsegcod and cmtse10.tipsegcod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cmtse10;
SELECT COUNT(*) FROM cmtse10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbtae10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM cmtse10", con=connection3)









connection3.close()


Cantidad de filas en la tabla cmtse10 antes de la inserción: 36
Cantidad de filas en la tabla mbtae10 después de la inserción: 36
La cantidad de filas insertadas fue de: 0


In [14]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN tipsegcod TYPE character(2),
ALTER COLUMN tipsegdes TYPE character(40),
ALTER COLUMN indpagprescod TYPE character(1),
ALTER COLUMN tipsegcalvbp TYPE character(1),
ALTER COLUMN indotorcittcod TYPE character(1),
ALTER COLUMN estregcod TYPE character(1),
ALTER COLUMN indplansaluflg TYPE character(1),
ALTER COLUMN tipsegdecurgflg TYPE character(1),
ALTER COLUMN tipsegusucrecod TYPE character(10),
ALTER COLUMN tipsegcrefec TYPE date USING tipsegcrefec::date,
ALTER COLUMN tipsegmodfec TYPE date USING tipsegmodfec::date,
ALTER COLUMN tipsegusumodcod TYPE character(10);
INSERT INTO {dim} 

(cod_tse,des_tse) 

SELECT 


tipsegcod,tipsegdes

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.cod_tse = t.tipsegcod
);
"""

c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_tipseg antes de la inserción: 36
Cantidad de filas en la tabla dim_tipseg después de la inserción: 36
La cantidad de filas insertadas fue de: 0
